# 🔍 04. Bayesian Hyperparameter Optimization with Optuna
**Project**: XGBoost-Powered PE Malware Detection  
**Purpose**: Maximize Validation ROC-AUC using Tree-structured Parzen Estimators (TPE) with early pruning.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import optuna

sys.path.insert(0, str(Path('../').resolve()))
from utils.preprocessing import MalwarePreprocessor, prepare_splits, compute_scale_pos_weight, get_interaction_constraints, ALL_FEATURES
from utils.optuna_tuner import tune_xgboost, get_optuna_history

optuna.logging.set_verbosity(optuna.logging.WARNING)

df = pd.read_csv('../data/synthetic_malware_data.csv')
train_df, val_df, test_df = prepare_splits(df)

preprocessor = MalwarePreprocessor(scaler_type='robust')
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)

y_train = train_df['label'].values
y_val = val_df['label'].values
spw = compute_scale_pos_weight(train_df['label'])
constraints = get_interaction_constraints(ALL_FEATURES)


## 1. Execute Optimization Study (25 Trials Demo)


In [ ]:
best_params, study = tune_xgboost(
    X_train, y_train, X_val, y_val,
    interaction_constraints=constraints,
    scale_pos_weight=spw,
    n_trials=25
)
print("Optimal Hyperparameters found:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Best Validation AUC: {study.best_value:.4f}")


## 2. Optuna Optimization History & Parameter Importance


In [ ]:
history = get_optuna_history(study)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(history['trial_numbers'], history['values'], 'o-', alpha=0.6, label='Trial AUC')
plt.plot(history['trial_numbers'], history['best_values'], 'r-', linewidth=2, label='Best AUC so far')
plt.title('Optuna Optimization Trajectory (ROC-AUC)', fontsize=14)
plt.xlabel('Trial Number')
plt.ylabel('Validation ROC-AUC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
